In [4]:
%pip install kagglehub
import kagglehub
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

Note: you may need to restart the kernel to use updated packages.


In [5]:
path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\aazee\.cache\kagglehub\datasets\yeanzc\telco-customer-churn-ibm-dataset\versions\1


In [6]:
# Excel File Download Path
# Excel File Download Path
%pip install openpyxl

excel_files = [f for f in os.listdir(path) if f.endswith('.xlsx')]
filepath = os.path.join(path, excel_files[0])
print("Reading file:", filepath)

df = pd.read_excel(filepath, engine='openpyxl')

raw_dir = os.path.join('..', 'data', 'raw')
os.makedirs(raw_dir, exist_ok=True)

raw_path = os.path.join(raw_dir, 'raw_telco_churn.csv')
df.to_csv(raw_path, index=False)
print(f"Raw dataset exported as '{raw_path}'")

print("Data shape (rows, columns):", df.shape)
df.head()

excel_files = [f for f in os.listdir(path) if f.endswith('.xlsx')]
filepath = os.path.join(path, excel_files[0])
print("Reading file:", filepath)

df = pd.read_excel(filepath)
raw_dir = os.path.join('..', 'data', 'raw')
os.makedirs(raw_dir, exist_ok=True)

raw_path = os.path.join(raw_dir, 'raw_telco_churn.csv')
df.to_csv(raw_path, index=False)
print(f"Raw dataset exported as '{raw_path}'")

print("Data shape (rows, columns):", df.shape)

df.head()


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [7]:
print("=== DATAFRAME INFO ===")
df.info()

print("\n=== CHURN DISTRIBUTION ===")
print(df['Churn Value'].value_counts())
print("\n=== CHURN PERCENTAGE ===")
print(df['Churn Value'].value_counts(normalize=True) * 100)

=== DATAFRAME INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Ser

In [8]:
# Blank Strings to NaN to 0
blank_count = (df['Total Charges'] == ' ').sum()
print(f"Blank 'Total Charges' entries found: {blank_count}")

df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

df['Total Charges'] = df['Total Charges'].fillna(0)

print("Updated 'Total Charges' data type:", df['Total Charges'].dtype)

Blank 'Total Charges' entries found: 11
Updated 'Total Charges' data type: float64


In [9]:
# Columns that leak target information or act as raw identifiers
cols_to_drop = [ 'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Label', 'Churn Reason', 'Churn Score', 'CLTV' ]
# Drop administrative & leakage columns
df_cleaned = df.drop(columns=cols_to_drop, errors='ignore')
print("Remaining features for modeling:", df_cleaned.shape[1])
print("Columns list:\n", df_cleaned.columns.tolist())

Remaining features for modeling: 20
Columns list:
 ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value']


In [10]:
# Create Tenure Cohorts
def get_tenure_cohort(months):
  if months <= 12:
    return 'Cohort_0_12'
  elif months <= 24:
    return 'Cohort_13_24'
  elif months <= 48:
    return 'Cohort_25_48'
  else:
    return 'Cohort_49_72'

df_cleaned['Tenure_Cohort'] = df_cleaned['Tenure Months'].apply(get_tenure_cohort)

# Total Subscribed Services
service_cols = [ 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies' ]

# No. of 'Yes' Service
df_cleaned['Total_Services'] = (df_cleaned[service_cols] == 'Yes').sum(axis=1)

# Monthly-to-Tenure Ratio
df_cleaned['Monthly_to_Tenure_Ratio'] = df_cleaned['Monthly Charges'] / (df_cleaned['Tenure Months'] + 1)
print("Engineered Features Added Successfully!")
print("New DataFrame shape:", df_cleaned.shape)
df_cleaned[['Tenure Months', 'Tenure_Cohort', 'Total_Services', 'Monthly_to_Tenure_Ratio']].head()

Engineered Features Added Successfully!
New DataFrame shape: (7043, 23)


,Tenure Months,Tenure_Cohort,Total_Services,Monthly_to_Tenure_Ratio
0,2,Cohort_0_12,2,17.950000
1,2,Cohort_0_12,0,23.566667
2,8,Cohort_0_12,3,11.072222
3,28,Cohort_25_48,4,3.613793
4,49,Cohort_49_72,4,2.074000


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

# 0. Separate Features (X) and Target (y), then Split
X = df_cleaned.drop(columns=['Churn Value'])
y = df_cleaned['Churn Value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 1. Define ALL Categorical Features (including remaining object columns)
nominal_features = [
    'Contract', 'Payment Method', 'Internet Service',
    'Multiple Lines', 'Online Security', 'Online Backup',
    'Device Protection', 'Tech Support', 'Streaming TV',
    'Streaming Movies', 'Tenure_Cohort'
]
binary_features = [
    'Gender', 'Senior Citizen', 'Partner', 'Dependents',
    'Phone Service', 'Paperless Billing'
]
categorical_cols = nominal_features + binary_features

# 2. Define Continuous Numerical Features for Scaling
numeric_cols = [
    'Tenure Months', 'Monthly Charges', 'Total Charges',
    'Total_Services', 'Monthly_to_Tenure_Ratio'
]

# 3. Construct Unified Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# 4. Fit strictly on Train set & transform both Train and Test (Uppercase X!)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get clean feature names for the exported CSVs
feature_names = preprocessor.get_feature_names_out()
train_df = pd.DataFrame(X_train_processed, columns=feature_names)
train_df['Churn Value'] = y_train.values

test_df = pd.DataFrame(X_test_processed, columns=feature_names)
test_df['Churn Value'] = y_test.values

output_dir = os.path.join('..', 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)

# 5. Save the unified preprocessor artifact
preprocessor_path = os.path.join(output_dir, 'preprocessor.pkl')
joblib.dump(preprocessor, preprocessor_path)
print(f"\nSaved unified preprocessor to '{preprocessor_path}'!")

# 6. Export clean CSV files
train_path = os.path.join(output_dir, 'train_telco_churn.csv')
test_path = os.path.join(output_dir, 'test_telco_churn.csv')

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("\n=== FINAL DELIVERABLES EXPORTED ===")
print(f"- {train_path}:", train_df.shape)
print(f"- {test_path}: ", test_df.shape)


Saved unified preprocessor to '..\data\processed\preprocessor.pkl'!

=== FINAL DELIVERABLES EXPORTED ===
- ..\data\processed\train_telco_churn.csv: (5634, 36)
- ..\data\processed\test_telco_churn.csv:  (1409, 36)
